## 📦 Step 1: Install Required Packages

Run this cell to install the required libraries:
- `langchain-groq`: LangChain integration for high-speed Groq LLM inference.
- `langgraph`: Workflow orchestration and stateful agent memory.
- `langchain-core`: Base message schemas (`HumanMessage`, `AIMessage`, `SystemMessage`).
- `SpeechRecognition`: Captures microphone audio and converts speech to text (STT).
- `pyttsx3`: Fast, offline Text-to-Speech (TTS) engine for Python.
- `pyaudio`: Required by SpeechRecognition to access system microphones.
- `python-dotenv`: Automatically loads API keys from your `.env` file.

In [ ]:
# Install required packages
%pip install -q langchain-groq langgraph langchain-core SpeechRecognition pyttsx3 pyaudio python-dotenv

## 🔑 Step 2: Environment Setup & Groq API Key

You need a Groq API key to access `openai/gpt-oss-120b`.
- You can create a free key at [console.groq.com/keys](https://console.groq.com/keys).
- Create a `.env` file in this directory with `GROQ_API_KEY=your_key_here` or enter it securely below.

In [1]:
import os
import getpass
from dotenv import load_dotenv

# Load environment variables from .env file if it exists
load_dotenv()

# If not found in .env, prompt the user securely
if "GROQ_API_KEY" not in os.environ or not os.environ["GROQ_API_KEY"]:
    api_key = getpass.getpass("Enter your Groq API Key: ")
    os.environ["GROQ_API_KEY"] = api_key

print("✅ Groq API Key is configured and ready!")

✅ Groq API Key is configured and ready!


## 📚 Step 3: Imports

Import all required modules from Python, LangChain, LangGraph, and the audio libraries.

In [2]:
import os
import time
from typing import Annotated, TypedDict

# Audio libraries for STT and TTS
import speech_recognition as sr
import pyttsx3

# LangChain and LangGraph imports
from langchain_groq import ChatGroq
from langchain_core.messages import HumanMessage, SystemMessage, AIMessage, BaseMessage
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langgraph.checkpoint.memory import MemorySaver

print("✅ All modules imported successfully!")

ModuleNotFoundError: No module named 'speech_recognition'

## 🔊 & 🎤 Step 4: Audio Helpers (STT & TTS)

Here we implement two functions to handle audio I/O:
1. **`speak(text)`**: Converts text into speech and plays it out loud through your speakers using `pyttsx3`.
2. **`listen()`**: Calibrates for ambient background noise, listens to your voice through the microphone, and transcribes it to text using Google Web Speech Recognition (free, no extra API key needed).

In [3]:
def speak(text: str):
    """
    Convert text to speech and play it through the speaker.
    Uses pyttsx3 for fast, offline speech synthesis.
    """
    print(f"\n🔊 Assistant: {text}")
    try:
        engine = pyttsx3.init()
        # Adjust speaking rate (words per minute)
        engine.setProperty("rate", 175)
        # Adjust volume (0.0 to 1.0)
        engine.setProperty("volume", 1.0)
        engine.say(text)
        engine.runAndWait()
        engine.stop()
    except Exception as e:
        print(f"⚠️ Text-to-Speech Error: {e}")


def listen(timeout: int = 7, phrase_time_limit: int = 15) -> str | None:
    """
    Listen to microphone input and convert speech to text using SpeechRecognition.
    Returns transcribed string, or None if no speech was detected.
    """
    recognizer = sr.Recognizer()
    recognizer.dynamic_energy_threshold = True

    try:
        with sr.Microphone() as source:
            print("\n🎤 Listening... (Speak now)")
            # Calibrate recognizer for room ambient noise
            recognizer.adjust_for_ambient_noise(source, duration=0.8)
            audio = recognizer.listen(source, timeout=timeout, phrase_time_limit=phrase_time_limit)
            
            print("⏳ Transcribing speech...")
            # Transcribe audio using Google Web Speech Recognition
            transcription = recognizer.recognize_google(audio, language="en-US")
            print(f"👤 You said: \"{transcription}\"")
            return transcription.strip()
            
    except sr.WaitTimeoutError:
        print("⌛ Listening timed out: No speech detected.")
        return None
    except sr.UnknownValueError:
        print("❓ Speech was unclear or unintelligible. Please try again.")
        return None
    except sr.RequestError as e:
        print(f"⚠️ Speech Recognition service error: {e}")
        return None
    except Exception as e:
        print(f"⚠️ Audio device error: {e}")
        return None

## 🧠 Step 5: LangGraph Agent Setup with Memory

We configure the LangGraph workflow:
1. **`AgentState`**: A TypedDict with `messages: Annotated[list[BaseMessage], add_messages]` so new conversation turns are automatically appended.
2. **`ChatGroq`**: Configured with model `"openai/gpt-oss-120b"`.
3. **`chatbot_node`**: Guides the LLM to give concise, natural responses (1-3 sentences) suitable for voice audio.
4. **`MemorySaver`**: In-memory checkpointing that retains conversation context across multiple turns.

In [ ]:
# 1. Define Agent State schema
class AgentState(TypedDict):
    # add_messages handles appending new user/AI messages to history
    messages: Annotated[list[BaseMessage], add_messages]


# 2. Initialize the Groq Chat Model (openai/gpt-oss-120b)
llm = ChatGroq(
    model="openai/gpt-oss-120b",
    temperature=0.7,
)

# 3. System Prompt tailored for voice conversations
SYSTEM_PROMPT = SystemMessage(
    content=(
        "You are a helpful, friendly, and natural AI voice assistant. "
        "Keep your responses concise, conversational, and direct (1 to 3 sentences maximum), "
        "because your responses are spoken out loud to the user over speakers. "
        "Do not use markdown formatting such as bolding, asterisks, or bullet points in your speech."
    )
)

# 4. Node function for processing conversation
def chatbot_node(state: AgentState) -> AgentState:
    messages = state["messages"]
    # Prepend system prompt to guide the tone of the voice response
    full_prompt = [SYSTEM_PROMPT] + list(messages)
    response = llm.invoke(full_prompt)
    return {"messages": [response]}


# 5. Build and compile the LangGraph workflow with Checkpoint Memory
workflow = StateGraph(AgentState)
workflow.add_node("chatbot", chatbot_node)
workflow.add_edge(START, "chatbot")
workflow.add_edge("chatbot", END)

# MemorySaver persists conversation history across multiple turns
memory = MemorySaver()
agent_app = workflow.compile(checkpointer=memory)

print("✅ LangGraph Agent successfully compiled with MemorySaver!")

## 🔄 Step 6: Continuous Voice Conversation Loop (`while True`)

This cell runs the complete voice loop:
- Greets you via voice over your speakers.
- Runs a continuous `while True` loop listening for your voice.
- If you say `"exit"`, `"quit"`, `"stop"`, or `"goodbye"`, the loop ends gracefully.
- Otherwise, it invokes the LangGraph agent, retains conversation memory, and speaks back the answer.

> 🛑 **To stop manually**: Say `"exit"` or `"quit"`, or click the Jupyter **Stop (■)** button.

In [ ]:
def run_voice_agent(thread_id: str = "voice_session_1"):
    """
    Continuous voice conversation loop:
    Microphone -> Speech-to-Text -> LangGraph (GPT-OSS-120B) -> Text-to-Speech -> Speaker
    """
    # Thread ID identifies and persists this specific conversation memory
    config = {"configurable": {"thread_id": thread_id}}
    
    # Keywords to exit the voice loop
    EXIT_KEYWORDS = {"exit", "quit", "stop", "bye", "goodbye"}
    
    print("\n" + "=" * 55)
    print("🎙️  VOICE AGENT ACTIVE & LISTENING")
    print("💡 Say 'exit', 'quit', or 'stop' at any time to end.")
    print("=" * 55)
    
    # Initial spoken greeting
    greeting = "Hello! I am your AI voice assistant powered by Groq. How can I help you today?"
    speak(greeting)
    
    try:
        while True:
            # 1. Capture microphone audio and convert to text (STT)
            user_input = listen()
            
            # If no speech was detected, continue listening
            if not user_input:
                continue
            
            # 2. Check for exit commands
            clean_input = user_input.lower().strip(".,!? ")
            words = set(clean_input.split())
            if clean_input in EXIT_KEYWORDS or (words & EXIT_KEYWORDS):
                farewell = "Goodbye! It was a pleasure chatting with you. Have a wonderful day!"
                speak(farewell)
                print("\n🛑 Voice conversation ended by user exit command.")
                break
            
            # 3. Pass user text into LangGraph agent with conversation memory
            human_message = HumanMessage(content=user_input)
            result = agent_app.invoke({"messages": [human_message]}, config=config)
            
            # 4. Extract latest AI response message
            ai_message = result["messages"][-1]
            response_text = ai_message.content
            
            # 5. Play response through speaker (TTS)
            speak(response_text)
            
    except KeyboardInterrupt:
        print("\n\n🛑 Voice agent stopped manually by user.")
        speak("Session terminated. Goodbye!")


# Launch the voice agent
run_voice_agent()

## 🧪 Step 7: (Optional) Text-Mode Testing

If you do not have a microphone connected or want to quickly test the LangGraph workflow, memory, and TTS output via keyboard typing, you can run this cell.

In [ ]:
def run_text_test(thread_id: str = "text_test_session"):
    """Fallback interactive loop using keyboard typing while keeping audio TTS output."""
    config = {"configurable": {"thread_id": thread_id}}
    print("Type your message below (type 'exit' to quit):\n")
    
    while True:
        user_input = input("👤 You: ")
        if user_input.lower().strip() in {"exit", "quit", "stop"}:
            speak("Goodbye from text mode!")
            print("Stopped text test.")
            break
        
        result = agent_app.invoke({"messages": [HumanMessage(content=user_input)]}, config=config)
        response = result["messages"][-1].content
        speak(response)

# Uncomment the line below to test in text mode:
# run_text_test()